# Track B — Qwen2.5-VL-3B QLoRA SFT (Kaggle T4)

선행: (1) `snuai_code`(pack_code.py 산출 zip) Dataset attach, (2) 대회 데이터 attach, (3) Accelerator=GPU T4 x2.

실행 순서: 설치 → 경로/스모크 → 50-step 타이밍 프로브 → 본 학습. 세션 12h 제한 주의.

In [ ]:
# 1) 설치 — Unsloth가 호환 transformers/peft/trl/bitsandbytes 조합을 스스로 정렬한다.
!pip install -q unsloth imagehash
import subprocess, sys
with open('/kaggle/working/pip_freeze_train.txt','w') as f:
    f.write(subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True).stdout)
print('pip freeze saved (본선 재현성 증빙)')

In [ ]:
# 2) 코드 경로 + 대회 데이터 자동 탐지 (수동 수정 불필요)
import os, sys, glob
CODE = glob.glob('/kaggle/input/*/src')[0].rsplit('/src', 1)[0]  # snuai_code Dataset 루트
sys.path.insert(0, CODE)
print('inputs:', os.listdir('/kaggle/input'))

hits = sorted(glob.glob('/kaggle/input/*/train.csv')
              + glob.glob('/kaggle/input/*/*/train.csv')
              + glob.glob('/kaggle/input/*/*/*/train.csv'))
assert hits, '대회 데이터가 attach되지 않았습니다 (Add Input에서 대회 데이터/업로드한 Dataset 선택)'
DATA_ROOT = os.path.dirname(hits[0])
print('DATA_ROOT:', DATA_ROOT)

yaml_text = '\n'.join([
    f'data_dir: {DATA_ROOT}',
    f'train_csv: {DATA_ROOT}/train.csv',
    f'test_csv: {DATA_ROOT}/test.csv',
    f'sample_submission: {DATA_ROOT}/sample_submission.csv',
    f'train_image_dir: {DATA_ROOT}/train',
    f'test_image_dir: {DATA_ROOT}/test',
    'models_dir: /kaggle/working/models',
    'outputs_dir: /kaggle/working/outputs',
    'reports_dir: /kaggle/working/reports',
])
open('/kaggle/working/paths.yaml', 'w').write(yaml_text)
os.environ['SNUAI_PATHS_CONFIG'] = '/kaggle/working/paths.yaml'
os.makedirs('/kaggle/working/outputs', exist_ok=True)

from src.data.loader import load_split
print('train rows:', len(load_split('train')), '| test rows:', len(load_split('test')))

In [ ]:
# 3) 10-step 스모크 (버전/포맷/증강 검증) — 장기 학습 전 필수
from cloud.train_unsloth import run
from src.data.loader import load_paths
SFT = f'{CODE}/outputs/sft_train.jsonl'
DATA_DIR = load_paths()['data_dir']
run(f'{CODE}/configs/sft_qwen.yaml', SFT, DATA_DIR, limit=32)  # limit로 초단축 스모크

In [ ]:
# 4) 본 학습 (2 epoch). save_steps 체크포인트가 output_dir에 남으므로
#    세션이 끊기면 이 노트북 output을 다음 세션 input으로 attach 후 resume=True.
lora_dir = run(f'{CODE}/configs/sft_qwen.yaml', SFT, DATA_DIR, resume=False)
print('LoRA saved:', lora_dir)

In [ ]:
# 5) 병합 저장 (추론 노트북에서 플레인 transformers로 로드하기 위해)
# ⚠️ save_pretrained_merged는 vision 모델에서 LoRA를 병합하지 않고 베이스만 저장
#    (unsloth#1352, 2026-07-05 Colab 실측) → peft 표준 merge_and_unload 사용.
#    저장 후 train 샘플 생성으로 병합 검증 필수 (colab 노트북 B4 셀 참조).
from unsloth import FastVisionModel
model, processor = FastVisionModel.from_pretrained(lora_dir, load_in_4bit=False)
merged = model.merge_and_unload()
merged.save_pretrained('/kaggle/working/outputs/qwen25vl3b_merged')
processor.save_pretrained('/kaggle/working/outputs/qwen25vl3b_merged')
print('merged model saved -> Dataset으로 저장 후 infer 노트북에 attach')